# Does imp_context actually find the important tokens?

The Spearman table in `colab_faithfulness_loo` was the wrong instrument. The ground truth is
**74% ties at zero**, and the label's real job is `KEEP_g = top_n(r)` — a top-n selector, not a
full ranking of 81 patches. A label that nails the top 5 and scrambles the 60 irrelevant ones
scores ~0 on Spearman while doing exactly what it is meant to do.

So: **precision@k**, plus the controls that separate real structure from the `relu` operator.

| | what it tests |
|---|---|
| precision@k | do the label's top-k patches overlap the ground truth's top-k? |
| **reverse** `relu(q - a)` | the intuition says the ANSWER reaches beyond the question. Reverse should be weaker. If it isn't, there is no asymmetry to exploit. |
| **rotated** `relu(a - rot90(q))` | same sparsity, same marginals, **wrong spatial alignment**. Isolates how much precision comes from `relu` alone. |
| **mismatched answer** (GPU) | recompute `imp_a` from ANOTHER example's answer. If a wrong answer scores the same, the structure is not about content. |

Reuses the cached `wearvqa_faithfulness.pt`, so the expensive 81-ablation ground truth is **not**
recomputed. Only the mismatched-answer section needs the model (~20 forwards, 1-2 min).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from scipy.stats import spearmanr, pearsonr

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

CACHE    = "/content/drive/MyDrive/wearvqa_faithfulness.pt"
MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
KS       = (5, 10, 12)

assert os.path.exists(CACHE), (
    f"{CACHE} not found - run colab_faithfulness_loo.ipynb first to build the ground truth")
data = torch.load(CACHE, weights_only=False)

L_v = data[0]["drops"].numel()
G   = int(round(math.sqrt(L_v)))
N   = len(data)

# PMI labels (the cache stores only the raw importances)
base_q = VS.make_baseline_loo([d["imp_q"] for d in data])
base_a = VS.make_baseline_loo([d["imp_a"] for d in data])
for i, d in enumerate(data):
    d["pmi_q"] = VS.pmi_scores(d["imp_q"], base_q[i])
    d["pmi_a"] = VS.pmi_scores(d["imp_a"], base_a[i])
    d["context"] = torch.relu(d["pmi_a"] - d["pmi_q"])

print(f"{N} examples | L_v={L_v} ({G}x{G}) | chance precision@k = "
      + ", ".join(f"k={k}: {k / L_v:.1%}" for k in KS))

## 2. Diagnostics before any comparison

Three things that decide how to read everything after them:

1. **Is the ground truth sink-contaminated?** If the top-drop patch is the same corner across
   unrelated images, the "ground truth" is measuring attention-sink removal, not image content.
2. **How correlated are `imp_q` and `imp_a`?** `imp_context` is their difference. At r=0.95 the
   difference is 5% signal on the difference of two noise fields.
3. **What is `imp_context`'s actual dynamic range?** The heatmaps are min-max stretched per panel,
   so a tiny residual renders just as saturated as a large effect.

In [ ]:
# --- 1. sink contamination -------------------------------------------------
peaks = [int(d["drops"].argmax()) for d in data]
mode, cnt = Counter(peaks).most_common(1)[0]
print(f"top-drop patch is #{mode} (row {mode // G}, col {mode % G}) in {cnt}/{N} examples")
print(f"   all peaks: {sorted(Counter(peaks).items(), key=lambda x: -x[1])[:6]}")
print(f"   -> if one corner patch dominates, that is the attention sink, not semantics\n")

SINK = {mode}
clean = [i for i, p in enumerate(peaks) if p not in SINK]
print(f"sink-dominated examples : {N - len(clean)}/{N}")
print(f"semantically clean       : {len(clean)}/{N}  (indices {clean})\n")

# --- 2. how much room does the subtraction have? ---------------------------
rs = [spearmanr(d["pmi_q"].numpy(), d["pmi_a"].numpy())[0] for d in data]
rp = [pearsonr(d["pmi_q"].numpy(), d["pmi_a"].numpy())[0] for d in data]
print(f"corr(pmi_q, pmi_a)  spearman {np.mean(rs):.3f}   pearson {np.mean(rp):.3f}")
print(f"   -> imp_context carries roughly {(1 - np.mean(rp)) * 100:.0f}% of the variance;"
      " the rest cancels\n")

# --- 3. dynamic range: is the heatmap stretching a tiny residual? -----------
d0 = data[0]
for name in ("pmi_q", "pmi_a", "context"):
    v = d0[name]
    print(f"{name:<9} min {v.min():+.3f}  max {v.max():+.3f}  "
          f"range {v.max() - v.min():.3f}  nonzero {int((v.abs() > 1e-9).sum())}/{L_v}")
print("   -> if context's range is far smaller, its heatmap is a stretched residual")

## 3. Precision@k, with the controls

In [ ]:
def topk_set(v, k):
    return set(torch.topk(v, k).indices.tolist())

def rot90(v, turns=1):
    return torch.rot90(v.reshape(G, G), turns, (0, 1)).flatten()

g = torch.Generator().manual_seed(0)

def label_set(d):
    """every candidate scored, including the controls."""
    pq, pa = d["pmi_q"], d["pmi_a"]
    gp = min(G - 1, int(d["gaze"]["y_norm"] * G)) * G + min(G - 1, int(d["gaze"]["x_norm"] * G))
    r0, c0 = divmod(gp, G)
    gaze_prox = torch.tensor([-math.hypot(i // G - r0, i % G - c0) for i in range(L_v)])
    r1, c1 = divmod(L_v // 2, G)
    center = torch.tensor([-math.hypot(i // G - r1, i % G - c1) for i in range(L_v)])
    return {
        "imp_q  raw":            d["imp_q"],
        "imp_q  pmi":            pq,
        "imp_a  raw":            d["imp_a"],
        "imp_a  pmi":            pa,
        "imp_context  (a-q)":    torch.relu(pa - pq),
        "CTRL reverse (q-a)":    torch.relu(pq - pa),
        "CTRL rotated q":        torch.relu(pa - rot90(pq)),
        "gaze proximity":        gaze_prox,
        "center (no image)":     center,
        "CTRL random":           torch.rand(L_v, generator=g),
    }

# only score examples where the ground-truth top-k is genuinely nonzero
# (with 74% ties at 0, topk on ties would return arbitrary patches)
results = {k: defaultdict(list) for k in KS}
usable = {}
for k in KS:
    n_ok = 0
    for d in data:
        if float(torch.topk(d["drops"], k).values[-1]) <= 1e-6:
            continue                       # ground-truth top-k is not well defined
        n_ok += 1
        gt = topk_set(d["drops"], k)
        for name, v in label_set(d).items():
            results[k][name].append(len(topk_set(v, k) & gt) / k)
    usable[k] = n_ok

for k in KS:
    chance = k / L_v
    print(f"\n=== precision@{k}   (chance {chance:.1%}, {usable[k]}/{N} examples usable)")
    print(f"{'label':<22}{'prec':>8}{'vs chance':>11}{'p':>8}")
    print("-" * 49)
    names = sorted(results[k], key=lambda n: -np.mean(results[k][n]))
    for name in names:
        a = np.array(results[k][name])
        se = a.std(ddof=1) / max(np.sqrt(len(a)), 1e-9)
        z = (a.mean() - chance) / se if se > 0 else 0.0
        from scipy.stats import norm
        p = 2 * (1 - norm.cdf(abs(z)))
        star = "  *" if p < 0.05 else ""
        print(f"{name:<22}{a.mean():>8.1%}{a.mean() / chance:>10.2f}x{p:>8.3f}{star}")

### Reading this

* `imp_context` well above chance **and** above both `CTRL rotated q` and `CTRL reverse`
  -> the visual impression was right, and the earlier Spearman was simply blind to it.
* `imp_context` above chance but **matched by `CTRL rotated q`** -> the precision comes from the
  `relu` sparsity, not from the two maps disagreeing in the right places.
* `CTRL reverse` as strong as `imp_context` -> there is no answer-beyond-question asymmetry here.
* nothing above chance -> the labels do not locate the patches the answer needed.

## 4. Mismatched-answer control (needs the GPU)

Recompute `imp_a` on the **same image and question** but with **another example's answer**, then
rebuild `imp_context` from it. The ground-truth drops are unchanged and reused from cache.

If a wrong answer produces the same precision@k, the structure is coming from the operator and not
from what the answer actually needed.

In [ ]:
model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer

def build_inputs(image, question, answer):
    messages = [{"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

def raw_scores_for(inp):
    patched = S._patch_eager_globals(S._make_raw_capturing_eager(None))
    try:
        with torch.no_grad():
            model(**inp)
    finally:
        S._unpatch_eager_globals(patched)
    L = int(inp["input_ids"].shape[1])
    raw = {}
    for m in model.modules():
        r = getattr(m, "_raw_attn_scores", None)
        if r is not None and r.shape[-1] == L and r.shape[-2] == L:
            raw[int(getattr(m, "layer_idx", len(raw)))] = r[0].float()
        for attr in ("_raw_attn_scores", "_post_attn"):
            if hasattr(m, attr):
                delattr(m, attr)
    return raw

def answer_importance(img_path, question, answer):
    """imp over ANSWER-token rows for an arbitrary answer string."""
    img = S.load_image(img_path)
    inp, n_prompt = build_inputs(img, question, answer)
    ids = inp["input_ids"][0].cpu()
    img_id = S._find_image_token_id(model, processor)
    pad = tokenizer.pad_token_id
    img_mask = ids == img_id
    txt_mask = (ids != img_id) & (ids != (pad if pad is not None else -10**9))
    raw = raw_scores_for(inp)
    maps, tpos, _ = RS.sliced_maps_from_full(raw, img_mask, txt_mask)
    tt = tokenizer.convert_ids_to_tokens(ids[tpos].tolist())
    is_ans = torch.tensor([int(p) >= n_prompt for p in tpos.tolist()])
    m = RS.content_text_mask(tt, tokenizer) & is_ans
    if int(m.sum()) == 0:
        m = is_ans
    imp, *_ = VS.image_importance(maps, m)
    del raw, maps
    return imp

OFFSET = 7          # example i gets example (i+7)%N's answer -> different image AND type
fake = []
for i, d in enumerate(data):
    j = (i + OFFSET) % N
    fake.append(answer_importance(d["img_path"], d["question"], data[j]["answer"]))
    if (i + 1) % 5 == 0:
        print(f"  {i + 1}/{N}")

base_f = VS.make_baseline_loo(fake)
for i, d in enumerate(data):
    d["context_fake"] = torch.relu(VS.pmi_scores(fake[i], base_f[i]) - d["pmi_q"])

print("\n=== real vs mismatched answer")
print(f"{'k':<5}{'imp_context':>14}{'MISMATCHED':>14}{'chance':>10}")
print("-" * 43)
for k in KS:
    real, mism = [], []
    for d in data:
        if float(torch.topk(d["drops"], k).values[-1]) <= 1e-6:
            continue
        gt = topk_set(d["drops"], k)
        real.append(len(topk_set(d["context"], k) & gt) / k)
        mism.append(len(topk_set(d["context_fake"], k) & gt) / k)
    print(f"{k:<5}{np.mean(real):>13.1%}{np.mean(mism):>14.1%}{k / L_v:>10.1%}")
print("\nif the two columns match, the structure is relu(a-q), not the answer's content")

## 5. Restricted to the semantically clean examples

The sink-dominated examples have a ground truth that is measuring sink removal. On those, no
content-based label can score. This repeats precision@k on the subset whose top-drop patch is
**not** the sink.

In [ ]:
if len(clean) < 4:
    print(f"only {len(clean)} clean examples - too few to stratify; raise N_PER_TYPE and rerun "
          "colab_faithfulness_loo.ipynb")
else:
    k = 10
    chance = k / L_v
    print(f"precision@{k} on the {len(clean)} non-sink examples (chance {chance:.1%})\n")
    print(f"{'label':<22}{'clean':>9}{'all':>9}")
    print("-" * 40)
    rowsc = {}
    for name in label_set(data[0]):
        cl, al = [], []
        for i, d in enumerate(data):
            if float(torch.topk(d["drops"], k).values[-1]) <= 1e-6:
                continue
            gt = topk_set(d["drops"], k)
            v = len(topk_set(label_set(d)[name], k) & gt) / k
            al.append(v)
            if i in clean:
                cl.append(v)
        rowsc[name] = (np.mean(cl) if cl else float("nan"), np.mean(al))
    for name in sorted(rowsc, key=lambda n: -rowsc[n][0]):
        c, a = rowsc[name]
        print(f"{name:<22}{c:>9.1%}{a:>9.1%}")
    print("\nif imp_context jumps on the clean subset, sink contamination was masking it")

## 6. Verdict

Fill this in from section 3, 4 and 5 rather than from the heatmaps:

| observation | conclusion |
|---|---|
| `imp_context` > chance, > rotated, > reverse, and mismatched < real | the signal is real. Use `imp_context` as the teacher label, and fix the ground truth by excluding the sink. |
| `imp_context` ~ `CTRL rotated q` | the sparsity is the operator. The heatmaps were a `relu` artifact plus min-max stretching. |
| mismatched ~ real | the label does not depend on the answer's content. `imp_answer` is not carrying answer-specific information on this data. |
| nothing above chance anywhere | attention over this model's decoder does not locate the patches the answer needed. Switch the teacher to attention-rollout or LOO-derived labels, or change dataset. |

Whatever the outcome, the **sink** finding stands on its own and needs fixing in both places: the
labels correct for it, the LOO ground truth does not, so they disagree at the largest value in the
vector. Excluding the sink from both sides is the first correction to make.